# Scene Graphs, Groups, and Transforms

**Part I · Visualization** — Tutorial 07

Organize scenes as node hierarchies. Build parent/child trees with `VizGroup`
and mutate nodes by reference with `VizObjectRef`. You will learn to:

- Create groups (`add_group`) and entities inside groups (`new()` / `viz(...)`).
- Mutate nodes via `.entity` / `.style` / `.color` / `.opacity`.
- Apply per-object transforms (`translate`, `rotate`, `scale_by`,
  `set_transform`) and operator-based transforms (`Rotor` / `Motor` /
  `Translator` / `Dilator`).
- Manage labels and overlay nodes (`attach_to`).


## Setup


In [ ]:
from pytanga.geometry import Direction, Line, Point
from pytanga.geometry.operators import Rotor, Translator
from pytanga.viz import LabelStyle, LineStyle, PointStyle, Visualizer


## 1. Groups and children

`add_group()` creates an empty `VizGroup` node (a `THREE.Group`). Create
entities inside it with `group.new(...)` — or use the `viz(...)` shorthand for
`viz.new(...)` at the top level. Both return a `VizObjectRef`.


In [ ]:
viz = Visualizer(title="Scene graph — groups", add_default_axes=False, add_default_grid=False)

grp = viz.add_group("spinner")                 # returns a VizObjectRef
grp.new(Point(0, 0, 0), color="#ff4444", label="hub")
grp.new(
    Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)),
    color="#44aaff",
)

viz.flush()
viz.display_snapshot()


## 2. Mutating nodes by reference — `VizObjectRef`

A `VizObjectRef` wraps a node and lets you mutate it without tracking raw ids.
Property setters mark the correct dirty aspect automatically.


In [ ]:
viz = Visualizer(title="Scene graph — refs", add_default_axes=False, add_default_grid=False)

ref = viz.new(Point(1, 2, 3), color="#ff4444", label="P")

ref.entity = Point(4, 5, 6)      # replaces geometry (same kind) → "content" aspect
ref.color = "#00ff00"            # → "style" aspect
ref.opacity = 0.5                # → "style" aspect
ref.style = PointStyle(size=0.2)  # merges non-None style fields → "style"

# Labels attached to this node:
print("label ids:", ref.label_ids)
ref.update_label(text="moved")

viz.flush()
viz.display_snapshot()


## 3. Per-object transforms

Scene nodes carry a canonical transform (position + Euler `"XYZ"` rotation +
scale). `translate`, `rotate`, `scale_by`, and `set_transform` update it in
place.


In [ ]:
viz = Visualizer(title="Scene graph — transforms", add_default_axes=False, add_default_grid=False)

ref = viz.new(Point(0, 0, 0), color="#ff4444", label="P", style=PointStyle(size=0.15))

ref.translate(2, 0, 0)               # or translate(Point(...) / Direction(...) / Translator(...))
ref.rotate(angle=0.5, axis=(0, 0, 1))  # axis-angle, Euler "XYZ"
ref.scale_by(1.5)                    # uniform (or component-wise: scale_by(1, 2, 1))
ref.set_transform(position=(2, 0, 1), rotation=(0, 0, 0.5), scale=(1, 1, 1))

viz.flush()
viz.display_snapshot()


## 4. Operator-based transforms

`apply_transform()` composes an operator in local space: `Rotor`, `Motor`,
`Translator`, or `Dilator`.


In [ ]:
viz = Visualizer(title="Scene graph — operator transforms", add_default_axes=False, add_default_grid=False)

ref = viz.new(Point(1, 0, 0), color="#ff4444", label="P", style=PointStyle(size=0.15))

ref.apply_transform(Rotor(angle=0.9, axis=Direction(0, 0, 1)))
ref.apply_transform(Translator(vector=Direction(0, 0, 1)))

viz.flush()
viz.display_snapshot()


## 5. Aspect-scoped updates

Each node tracks which *aspects* changed; `flush()` emits a partial
`object_update` patch:

| Aspect | Payload | Effect |
|---|---|---|
| `full` | complete node dict | create/replace |
| `style` | `{"style": …}` | merge style, re-apply materials |
| `transform` | `{position, rotation, scale}` | apply to the `Object3D` in place |
| `content` | kind + geometry + style | update the inner mesh in place (same kind) |

Rotating a `VizGroup` therefore emits a single `transform` patch — children are
**not** re-serialized and their vertices are never recomputed. This is the key
to cheap in-place group animation.


## 6. Overlay nodes via `attach_to`

Overlay nodes (labels, annotations, titles) live in the screen/CSS plane and can
follow a referenced scene node with `attach_to`.


In [ ]:
from pytanga.viz import Label

viz = Visualizer(title="Scene graph — overlay", add_default_axes=False, add_default_grid=False)

point = viz.new(Point(0, 0, 0), color="#ff4444", label="P")

# A standalone label that follows the point in the CSS plane:
lbl = viz.new(Label(text="tracked", position=(0, 0.5, 0)), color="#ffffff")
lbl.attach_to = point.id

viz.flush()
viz.display_snapshot()


## Visual Examples

A two-link arm built from nested `VizGroup`s, with each rod rotating about its
own pivot. Export a static snapshot after setting the group rotations.


In [ ]:
import math

L1, L2 = 2.0, 1.5
viz = Visualizer(title="Scene graph — two-link arm", add_default_axes=False, add_default_grid=False)

arm1 = viz.add_group("arm1")
arm1.new(Point(0, 0, 0), color="#ffaa00", label="pivot1", style=PointStyle(size=0.10))
arm1.new(
    Line.from_points(Point(0, 0, 0), Point(L1, 0, 0)),
    color="#ff5555", label="rod1", label_style=LabelStyle(along=0.5), style=LineStyle(thickness=3.0),
)

arm2 = arm1.add_group("arm2")
arm2.set_transform(position=(L1, 0.0, 0.0))
arm2.new(Point(0, 0, 0), color="#ffaa00", label="pivot2", style=PointStyle(size=0.09))
arm2.new(
    Line.from_points(Point(0, 0, 0), Point(L2, 0, 0)),
    color="#5599ff", label="rod2", label_style=LabelStyle(along=0.5), style=LineStyle(thickness=3.0),
)
arm2.new(Point(L2, 0, 0), color="#44ff44", label="tip", style=PointStyle(size=0.10))

# Pose the arm (static snapshot — in a live scene, update these per frame).
arm1.set_transform(rotation=(0.0, 0.0, 0.6))
arm2.set_transform(rotation=(0.0, 0.0, -0.8))

viz.flush()
viz.display_snapshot()


## Summary

| Task | API |
|---|---|
| Create a group | `viz.add_group("name")` / `grp.add_group("name")` |
| Add inside a group | `grp.new(entity, ...)` |
| Top-level shorthand | `viz(entity, ...)` == `viz.new(...)` |
| Replace geometry | `ref.entity = new_entity` |
| Style / color / opacity | `ref.style` / `ref.color` / `ref.opacity` |
| Per-object transform | `translate` / `rotate` / `scale_by` / `set_transform` |
| Operator transform | `apply_transform(Rotor(...) / Translator(...) / …)` |
| Label management | `ref.label_ids` / `ref.labels` / `ref.update_label(...)` |
| Overlay follow | `overlay.attach_to = scene_node_id` |

**Next:** [08 — Styles & Colors](../08_styles_colors/).
